# GaAs/AlAs (110) Strained Superlattice

## 0. Introduction

This tutorial builds the GaAs/AlAs (110) superlattice and its two bulk reference cells from the publication:

> **Yoyo Hinuma, Andreas Grüneis, Georg Kresse and Fumiyasu Oba**, "Band alignment of semiconductors from
> density-functional theory and many-body perturbation theory", Physical Review B 90, 155405 (2014).
> [DOI: 10.1103/PhysRevB.90.155405](https://doi.org/10.1103/PhysRevB.90.155405)

Both materials are strained in plane to the average of their two lattice constants. The superlattice stacks GaAs and
AlAs along [110], without vacuum, so it has two identical interfaces. The paper uses 11 atomic layers of each
material; the slab builder counts (110) layers in pairs of atomic planes, so this notebook uses 12 planes of each
(6 layers).

The valence band offset is computed from this superlattice and from separate bulk cells of GaAs and of AlAs
strained as in the superlattice, so all three are saved:
[the simulation notebook](interface_3d_3d_gallium_arsenide_aluminum_arsenide_SIMULATION.ipynb) loads them by name.

## 1. Prepare the Environment
### 1.1. Set up the parameters

In [ ]:
SUBSTRATE_NAME = "mp-2534"  # GaAs, Standata
FILM_NAME = "mp-2172"  # AlAs, Standata
FILM_ELEMENT_REPLACEMENT = ("Ga", "Al")  # Standata cells of GaAs and AlAs come out in different zincblende settings; the film is built from the GaAs cell with Ga replaced by Al, at the AlAs lattice constant
MILLER_INDICES = (1, 1, 0)
NUMBER_OF_LAYERS = 6  # conventional (110) layers, two atomic planes each

SUPERLATTICE_NAME = "GaAs/AlAs (110) superlattice 12+12"
SUBSTRATE_BULK_NAME = "GaAs (110) bulk strained"
FILM_BULK_NAME = "AlAs (110) bulk strained"

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Load the bulk materials and strain them

Each bulk is strained in its conventional cubic cell to the average of the two lattice constants, before any slab is
cut: the interface builder re-creates its slabs from the bulk, so a strain applied to a slab would be lost.

In [ ]:
from mat3ra.esse.models.core.abstract.matrix_3x3 import Matrix3x3Schema
from mat3ra.made.material import Material
from mat3ra.made.tools.analyze.lattice.helpers import get_material_with_conventional_lattice
from mat3ra.made.tools.operations.core.unary import strain
from mat3ra.standata.materials import Materials


def get_conventional_cell(name):
    return get_material_with_conventional_lattice(Material.create(Materials.get_by_name_first_match(name)))


def strain_to_lattice_constant(material, lattice_constant):
    ratio = lattice_constant / material.lattice.a
    return strain(material, Matrix3x3Schema(root=[[ratio, 0, 0], [0, ratio, 0], [0, 0, ratio]]))


substrate_conventional = get_conventional_cell(SUBSTRATE_NAME)
film_conventional = get_conventional_cell(FILM_NAME)
film_lattice_constant = film_conventional.lattice.a
film_conventional = strain_to_lattice_constant(substrate_conventional, film_lattice_constant)
film_conventional.basis.elements.values = [FILM_ELEMENT_REPLACEMENT[1] if element == FILM_ELEMENT_REPLACEMENT[0] else element
                                           for element in film_conventional.basis.elements.values]
mean_lattice_constant = (substrate_conventional.lattice.a + film_lattice_constant) / 2
misfit = (substrate_conventional.lattice.a - film_lattice_constant) / mean_lattice_constant

print(f"GaAs a = {substrate_conventional.lattice.a:.4f} Å, AlAs a = {film_lattice_constant:.4f} Å")
print(f"mean a = {mean_lattice_constant:.4f} Å, misfit = {misfit * 100:.2f} %")

substrate_bulk = strain_to_lattice_constant(substrate_conventional, mean_lattice_constant)
film_bulk = strain_to_lattice_constant(film_conventional, mean_lattice_constant)

## 2. Create the Superlattice
### 2.1. Create the (110) slabs, without vacuum

In [ ]:
from mat3ra.made.tools.helpers import create_slab

substrate_slab, film_slab = [
    create_slab(crystal=bulk, miller_indices=MILLER_INDICES, number_of_layers=NUMBER_OF_LAYERS,
                use_conventional_cell=True, vacuum=0.0)
    for bulk in (substrate_bulk, film_bulk)
]

### 2.2. Stack the slabs and reduce to the primitive cell

The slabs are joined directly, without gap or vacuum, so the second, periodic interface closes the cell.

In [ ]:
from mat3ra.made.tools.helpers import create_interface_simple_between_slabs

superlattice = create_interface_simple_between_slabs(substrate_slab=substrate_slab, film_slab=film_slab,
                                                     gap=None, vacuum=0.0, reduce_result_cell_to_primitive=True)

### 2.3. Create the bulk reference cells

One (110) layer of each strained bulk, periodic along the stacking axis, with the same in-plane lattice as the
superlattice. Atom labels are removed from all three materials: labels break the Quantum ESPRESSO input.

In [ ]:
substrate_bulk_reference, film_bulk_reference = [
    create_slab(crystal=bulk, miller_indices=MILLER_INDICES, number_of_layers=1,
                use_conventional_cell=True, vacuum=0.0)
    for bulk in (substrate_bulk, film_bulk)
]

superlattice.name = SUPERLATTICE_NAME
substrate_bulk_reference.name = SUBSTRATE_BULK_NAME
film_bulk_reference.name = FILM_BULK_NAME
materials = [superlattice, substrate_bulk_reference, film_bulk_reference]
for material in materials:
    material.basis.set_labels_from_list([])

### 2.4. Print the structure

Every atom has four nearest neighbors in the zincblende structure; the count below uses a cutoff between the first
and second neighbor shells.

In [ ]:
import numpy as np
from collections import Counter
from ase.neighborlist import neighbor_list
from mat3ra.made.tools.convert import to_ase

NEIGHBOR_CUTOFF = 0.5 * (np.sqrt(3) / 4 + np.sqrt(2) / 2) * mean_lattice_constant  # Å, between 1st and 2nd shell


def get_plane_spacings(material):
    cartesian_material = material.clone()
    cartesian_material.to_cartesian()
    heights = np.array(cartesian_material.basis.coordinates.values)[:, 2]
    plane_heights = np.unique(np.round(heights, 4) % round(material.lattice.c, 4))
    return np.diff(np.append(plane_heights, plane_heights[0] + material.lattice.c))


for material in materials:
    lattice = material.lattice
    coordination = Counter(np.bincount(neighbor_list("i", to_ase(material), NEIGHBOR_CUTOFF)))
    print(f"{material.name}: {dict(Counter(material.basis.elements.values))}, {material.basis.number_of_atoms} atoms, "
          f"cell {lattice.a:.4f} x {lattice.b:.4f} x {lattice.c:.4f} Å")
    print(f"  plane spacings along c: {np.unique(np.round(get_plane_spacings(material), 4)).tolist()} Å")
    print(f"  coordination numbers: {dict(coordination)}")

## 3. Visualize the Result(s)

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

visualize_materials([{"material": material, "title": material.name, "rotation": "-90x"} for material in materials],
                    repetitions=[2, 2, 1])

## 4. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.material import set_materials

set_materials(materials)

## References

[1] Y. Hinuma, A. Grüneis, G. Kresse and F. Oba, "Band alignment of semiconductors from density-functional theory and
many-body perturbation theory", Phys. Rev. B 90, 155405 (2014).
[DOI: 10.1103/PhysRevB.90.155405](https://doi.org/10.1103/PhysRevB.90.155405)